# Portable pinyin acoustic embeddings

Readable self-contained Kaggle inference producing the same validated bundle as the SDK. No project checkout or automatic package installation is required.


## Dependencies and provenance

NumPy is required. Real inference also needs Torch and Transformers; librosa handles resampling/non-WAV decoding and soundfile inspects MP3. Hashes identify the original SDK and adapted inline sections.


In [ ]:
from copy import deepcopy
from dataclasses import dataclass
from importlib.metadata import version
import csv
import hashlib
import importlib.metadata
import json
import os
import pickle
import platform
import re
import shutil
import tempfile
import uuid
import wave
from pathlib import Path

import numpy as np
if version("pypinyin") != "0.55.0":
    raise RuntimeError("pypinyin 0.55.0 is required")


In [ ]:
BUILD_METADATA = {'original_sdk_source_sha256': {'artifacts.py': '9447013a45a347b7df6735cde9dc7c40f0931d53792165999c482c384321d64d', 'data_contract.py': '96e5dd2a0390836d8892badb1baea513b83fb496fb1e4f48dd24ae6bb5229b4d', 'pinyin.py': '6d080de3613cd219edd5553512e17e4fe596a7bb881d59c0f6acd654bb85f1d3', 'preparation/acoustic_bundle.py': 'f349ab3257e4b7112d8a61637e1416fb833ec0e913a8f61f102accb19cabb1a1', 'preparation/config.py': '7874db38d63379cb809cb73f272f113a1f8a0789688a1acdf16df8cbc3bc7a38', 'preparation/embeddings.py': '2dfd1d57fd7610ba27fa6445299a505a776333765e408f305ce053272553fcd7', 'preparation/readers.py': '265736c8aac51eb4b6fedce54abb40cf9739acfbf6088bb76f25769e5f083bd7'}, 'inline_source_sha256': {'acoustic-validation': '98edfe53b1a76a4122cddc1f04ecf72eb5caffc03cf48cbd098f4cbf9668d2e2', 'artifact-helpers': '55452cc148ac690fec4164535346014f2e08f4a75dec9485f89af731bc22ba59', 'audio-model': '3b5a7071021cb9ba4fc9c76d5a32b07c34510655affa1b3362a058ee6ec626bb', 'bundle-creation': 'a94c32fce3317a6356735c15be99504d73327283047bcd7b66859b9c8dce69c8', 'bundle-validation': '1039d1064327be482818df0212ac1d01d476d910835d99a491664b29403fbce9', 'embedding-cache': 'd90e5a49ea1553147f8c958515ca39d764e50ba9c917475ca817485b903ce379', 'model-config': '1f85757af42a7fef988391efe2eaafc832437b43ac8d6c2bc815a08c7291a6a7', 'pinyin-contract': '068f53115ff99936d22177444a8b545b51195e188d0bd69230c930cd26976d67', 'sound-inventory': 'cf2f20796b9b7cf7b424f87531ca79251363d70bc8bd390815801256fc99a501'}}


## Pinyin contract

Canonical tones and decomposition use the fixed SDK inventories and zero-initial rules.


In [ ]:
"""Project TONE3 input contract backed by pinned pypinyin inventories."""
from dataclasses import dataclass
from hashlib import sha256
from importlib.metadata import version
import re

from pypinyin.constants import PINYIN_DICT
from pypinyin.contrib.tone_convert import to_finals, to_initials, to_normal

PINYIN_CONTRACT_VERSION = "pypinyin-tone3-1"


def _require_pypinyin_version():
    installed = version("pypinyin")
    if installed != "0.55.0":
        raise RuntimeError(f"pypinyin 0.55.0 required, found {installed}")
    return installed


_PYPINYIN_VERSION = _require_pypinyin_version()
# Dictionary readings contain tone marks; external project inputs do not.
VALID_SYLLABLES = frozenset(
    to_normal(reading, v_to_u=False).lower()
    for readings in PINYIN_DICT.values()
    for reading in readings.split(",")
)
_INPUT_SYLLABLES = tuple(sorted(base for base in VALID_SYLLABLES if re.fullmatch(r"[a-z]+", base)))
_STRICT_COMPONENTS = tuple(
    (to_initials(base, strict=True), to_finals(base, strict=True, v_to_u=False))
    for base in _INPUT_SYLLABLES
)
INITIALS = sorted({initial for initial, final in _STRICT_COMPONENTS if final})
FINALS = sorted({final for _, final in _STRICT_COMPONENTS if final})
TONES = [str(tone) for tone in range(1, 6)]
PARSER_PROVENANCE = {
    "library": "pypinyin", "version": _PYPINYIN_VERSION,
    "api": "pypinyin.contrib.tone_convert", "strict": True,
    "input_format": "lowercase-ascii-tone3", "umlaut_storage": "v",
    "input_alias": "u:", "missing_tone": "5", "empty_final": "reject",
    "inventory_source": "pypinyin.constants.PINYIN_DICT",
    "inventory_sha256": sha256("\n".join(sorted(VALID_SYLLABLES)).encode("utf-8")).hexdigest(),
    "vocabulary_sha256": sha256(repr((INITIALS, FINALS, TONES)).encode("utf-8")).hexdigest(),
}


@dataclass(frozen=True)
class PinyinComponents:
    initial: str
    final: str
    tone: str


def canonical_label(token: str, *, tone_convention="neutral_unmarked") -> str:
    """Accept one lowercase TONE3 syllable, allowing u: and unmarked neutral."""
    if tone_convention not in {"neutral_unmarked", "explicit"}:
        raise ValueError(f"unsupported tone convention: {tone_convention!r}")
    if not isinstance(token, str) or not token:
        raise ValueError(f"Pinyin must be a non-empty string: {token!r}")
    if re.fullmatch(r"(?:[a-z]|u:)+[1-5]?", token) is None:
        raise ValueError(f"invalid Pinyin format (lowercase ASCII TONE3 required): {token!r}")
    canonical = token.replace("u:", "v")
    if canonical[-1] not in TONES:
        if tone_convention == "explicit":
            raise ValueError(f"explicit tone required: {token!r}")
        canonical += "5"
    if re.fullmatch(r"[a-z]+[1-5]", canonical) is None:
        raise ValueError(f"invalid canonical Pinyin format: {token!r}")
    base = canonical[:-1]
    if base not in VALID_SYLLABLES:
        raise ValueError(f"syllable absent from pypinyin dictionary: {token!r}")
    initial = to_initials(base, strict=True)
    final = to_finals(base, strict=True, v_to_u=False)
    if not final or initial not in INITIALS or final not in FINALS:
        raise ValueError(f"unsupported strict Pinyin components (non-empty final required): {token!r}")
    return canonical


def validate_canonical_label(token: str) -> str:
    """Reject noncanonical stored labels instead of silently repairing them."""
    if canonical_label(token, tone_convention="explicit") != token:
        raise ValueError(f"stored Pinyin label is not canonical: {token!r}")
    return token


def parse_components(token: str, *, tone_convention="neutral_unmarked") -> PinyinComponents:
    canonical = canonical_label(token, tone_convention=tone_convention)
    base = canonical[:-1]
    return PinyinComponents(to_initials(base, strict=True), to_finals(base, strict=True, v_to_u=False), canonical[-1])


def encode_components(components: PinyinComponents) -> tuple[int, int, int]:
    try:
        return INITIALS.index(components.initial), FINALS.index(components.final), TONES.index(components.tone)
    except ValueError as exc:
        raise ValueError(f"components outside Pinyin vocabulary: {components!r}") from exc


def canonical_labels(tokens, *, tone_convention="neutral_unmarked"):
    return [canonical_label(token, tone_convention=tone_convention) for token in tokens]


def decompose(token: str, *, tone_convention="neutral_unmarked") -> tuple[int, int, int]:
    return encode_components(parse_components(token, tone_convention=tone_convention))


## Artifact helpers

Deterministic hashes, JSON/JSONL output, and safe relative artifact paths.


In [ ]:
def sha256_file(path: str | Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def write_json(path: str | Path, value) -> None:
    Path(path).write_text(json.dumps(value, ensure_ascii=False, sort_keys=True,
                                     indent=2) + "\n", encoding="utf-8")

def write_jsonl(path: str | Path, rows) -> None:
    Path(path).write_text("".join(json.dumps(row, ensure_ascii=False, sort_keys=True)
                                  + "\n" for row in rows), encoding="utf-8")

def read_jsonl(path: str | Path) -> list[dict]:
    result = []
    for number, line in enumerate(Path(path).read_text(encoding="utf-8").splitlines(), 1):
        try:
            value = json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(f"invalid JSONL at line {number}") from exc
        if not isinstance(value, dict):
            raise ValueError(f"JSONL row {number} is not an object")
        result.append(value)
    return result

def safe_artifact(root: str | Path, relative: str) -> Path:
    root = Path(root).resolve()
    rel = Path(relative)
    if rel.is_absolute() or ".." in rel.parts:
        raise ValueError(f"unsafe artifact path: {relative}")
    path = (root / rel).resolve()
    if path != root and root not in path.parents:
        raise ValueError(f"unsafe artifact path: {relative}")
    cursor = root
    for part in rel.parts:
        cursor = cursor / part
        if cursor.exists() and cursor.is_symlink():
            raise ValueError(f"symlink artifact path: {relative}")
    return path


## Vector validation

Vectors must be finite, nonzero float32 values with one shared dimension and exact inventory coverage.


In [ ]:
DATA_CONTRACT_VERSION = "prepared-1"

def validate_acoustic(mapping, *, expected_keys=None):
    import numpy as np

    if not isinstance(mapping, dict) or not mapping:
        raise ValueError("acoustic mapping must be a non-empty dict")
    if any(not isinstance(key, str) or not key for key in mapping):
        raise ValueError("invalid acoustic key")
    if expected_keys is not None and set(mapping) != set(expected_keys):
        raise ValueError("acoustic keys disagree with inventory")
    dimension = None
    for key, value in sorted(mapping.items()):
        vector = np.asarray(value, dtype=np.float32)
        if vector.ndim != 1 or not vector.size:
            raise ValueError(f"invalid acoustic vector: {key}")
        if dimension is None:
            dimension = vector.size
        if (vector.size != dimension or not np.isfinite(vector).all()
                or np.linalg.norm(vector.astype(np.float64)) <= 1e-12):
            raise ValueError(f"invalid acoustic vector: {key}")
    return dimension


## Model configuration

Revision and device are explicit; float32 and `mean_all` are supported, while unproven `mean_valid` masks are rejected.


In [ ]:
SCHEMA_VERSION = 1

def validate_model_config(model):
    _keys(model, {"id", "revision", "layer", "pooling", "sample_rate", "mono", "padding", "dtype", "device"}, {"processor"}, "model")
    if not isinstance(model["revision"], str) or not model["revision"]:
        raise ValueError("model revision is required")
    if model["pooling"] not in {"mean_all", "mean_valid"} or model["padding"] not in {"legacy_one_second", "none"}:
        raise ValueError("unsupported pooling or padding")
    if model["pooling"] == "mean_valid":
        raise ValueError("mean_valid is unsupported until padding masks can be proven correct")
    if model["dtype"] != "float32" or not isinstance(model["mono"], bool) or not isinstance(model["layer"], int) or isinstance(model["layer"], bool):
        raise ValueError("model requires dtype=float32, boolean mono, and integer layer")
    if not isinstance(model["device"], str) or not model["device"]:
        raise ValueError("model device must be explicit")
    if not isinstance(model["sample_rate"], int) or isinstance(model["sample_rate"], bool) or model["sample_rate"] <= 0:
        raise ValueError("sample_rate must be a positive integer")
    return deepcopy(model)

def _keys(value, required, optional, name):
    if not required.issubset(value) or set(value) - required - optional:
        raise ValueError(f"invalid {name} keys")


## Sound inventory

Read the pinned manifest, canonicalize labels, inspect complete audio, and hash every source file.


In [ ]:
def verify_source(path, expected=None):
    path = Path(path).resolve()
    if not path.exists():
        raise FileNotFoundError(path)
    digest = sha256_file(path) if path.is_file() else None
    if expected is not None and digest != expected:
        raise ValueError(f"source checksum mismatch: {path}")
    return path, digest

def read_sounds(spec):
    root = Path(spec["path"]).resolve()
    if not root.is_dir():
        raise FileNotFoundError(root)
    manifest_path = spec.get("manifest")
    entries = []
    if manifest_path:
        manifest, manifest_hash = verify_source(manifest_path, spec.get("sha256"))
        with manifest.open(encoding="utf-8-sig", newline="") as handle:
            reader = csv.DictReader(handle, strict=True)
            if reader.fieldnames != ["path", "pinyin"]:
                raise ValueError("sound manifest headers must be path,pinyin")
            entries = [(row["path"], row["pinyin"]) for row in reader]
        source_hash = manifest_hash
    else:
        files = sorted((p for p in root.iterdir() if p.is_file() and p.suffix.lower() in {".wav", ".mp3"}), key=lambda p: p.name)
        entries = [(p.name, p.stem) for p in files]
        source_hash = None
        if spec.get("sha256") is not None:
            raise ValueError("sounds sha256 requires a manifest file")
    inventory, seen = [], {}
    for relative, raw_label in entries:
        path = (root / relative).resolve()
        if root not in path.parents or not path.is_file() or path.suffix.lower() not in {".wav", ".mp3"}:
            raise ValueError(f"unsafe or unsupported sound path: {relative}")
        label = canonical_labels([raw_label], tone_convention=spec["tone_convention"])[0]
        if label in seen:
            raise ValueError(f"canonical sound collision: {label}: {seen[label]} and {relative}")
        seen[label] = relative
        info = audio_info(path)
        inventory.append({"label": label, "raw_label": raw_label, "path": str(path), "relative_path": relative,
                          "sha256": sha256_file(path), **info})
    if not inventory:
        raise ValueError("sound inventory is empty")
    return sorted(inventory, key=lambda row: (row["label"], row["relative_path"])), {"path": str(root), "sha256": source_hash}

def audio_info(path):
    if path.suffix.lower() == ".wav":
        with wave.open(str(path), "rb") as handle:
            rate, channels, frames, width = handle.getframerate(), handle.getnchannels(), handle.getnframes(), handle.getsampwidth()
            raw = handle.readframes(frames)
            if rate <= 0 or channels <= 0 or frames <= 0 or len(raw) != frames * channels * width:
                raise ValueError(f"invalid or truncated WAV: {path}")
            return {"sample_rate": rate, "channels": channels, "frames": frames, "duration_seconds": frames / rate}
    try:
        import soundfile as sf
        info = sf.info(str(path))
        return {"sample_rate": info.samplerate, "channels": info.channels,
                "frames": info.frames, "duration_seconds": info.duration}
    except ImportError as exc:
        raise RuntimeError("soundfile is required to inspect MP3 inputs") from exc


## Audio decoding and Wav2Vec backend

Decode lazily, mono-mix, resample, optionally pad to one second, select a hidden layer, and mean-pool. Heavy libraries load only on used paths.


In [ ]:
class TransformersBackend:
    def __init__(self, config):
        try:
            import torch
            from transformers import AutoModel, AutoProcessor
        except ImportError as exc:
            raise RuntimeError("torch and transformers are required for real inference") from exc
        self.torch = torch
        processor_id = config.get("processor") or config["id"]
        self.processor = AutoProcessor.from_pretrained(processor_id, revision=config["revision"])
        self.model = AutoModel.from_pretrained(config["id"], revision=config["revision"]).to(config["device"])
        self.model.eval()

    def __call__(self, entry, config):
        import numpy as np
        samples, sample_rate = _read_audio(Path(entry["path"]), mono=config["mono"])
        if sample_rate != config["sample_rate"]:
            try:
                import librosa
            except ImportError as exc:
                raise RuntimeError("librosa is required for resampling") from exc
            samples = librosa.resample(samples, orig_sr=sample_rate, target_sr=config["sample_rate"])
        if config["padding"] == "legacy_one_second" and len(samples) < config["sample_rate"]:
            samples = np.pad(samples, (0, config["sample_rate"] - len(samples)))
        inputs = self.processor(samples, sampling_rate=config["sample_rate"], return_tensors="pt",
                                padding=config["padding"] != "none")
        inputs = {key: value.to(config["device"]) for key, value in inputs.items()}
        with self.torch.inference_mode():
            output = self.model(**inputs, output_hidden_states=True)
        hidden = output.hidden_states[int(config["layer"])]
        if config["pooling"] == "mean_all":
            pooled = hidden.mean(dim=1)
        else:
            if "attention_mask" not in inputs or not hasattr(self.model, "_get_feature_vector_attention_mask"):
                raise ValueError("mean_valid requires model feature-vector attention-mask support")
            mask = self.model._get_feature_vector_attention_mask(hidden.shape[1], inputs["attention_mask"])
            pooled = (hidden * mask.unsqueeze(-1)).sum(dim=1) / mask.sum(dim=1, keepdim=True)
        return pooled[0].detach().cpu().to(self.torch.float32).numpy()

def _read_audio(path, *, mono):
    import numpy as np
    if path.suffix.lower() == ".wav":
        with wave.open(str(path), "rb") as handle:
            channels, width, rate = handle.getnchannels(), handle.getsampwidth(), handle.getframerate()
            raw = handle.readframes(handle.getnframes())
        if width != 2:
            raise ValueError("stdlib WAV reader supports only 16-bit PCM")
        values = np.frombuffer(raw, dtype="<i2").astype(np.float32) / 32768.0
        if channels > 1:
            values = values.reshape(-1, channels)
            if mono:
                values = values.mean(axis=1)
            else:
                raise ValueError("multi-channel input requires mono=true")
        return values, rate
    try:
        import librosa
    except ImportError as exc:
        raise RuntimeError("librosa is required for non-WAV audio") from exc
    return librosa.load(str(path), sr=None, mono=mono)


## Full-inventory cache

Cache identity binds audio bytes, all processing/model settings, and backend identity; every vector is validated.


In [ ]:
def cache_key(entry, model, backend_identity):
    payload = {"audio_sha256": entry["sha256"], "model": model, "backend": backend_identity}
    return hashlib.sha256(json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()).hexdigest()

def embed_inventory(inventory, model_config, cache_root, backend=None):
    import numpy as np

    cache_root = Path(cache_root).resolve()
    cache_root.mkdir(parents=True, exist_ok=True)
    result, computed, cached = {}, 0, 0
    backend_identity = getattr(backend, "cache_id", None) if backend is not None else "transformers"
    use_cache = backend is None or backend_identity is not None
    real_backend = backend
    for entry in inventory:
        key = cache_key(entry, model_config, backend_identity or "uncached-stub")
        cache_path = cache_root / f"{key}.pkl"
        if use_cache and cache_path.is_file():
            with cache_path.open("rb") as handle:
                vector = pickle.load(handle)
            cached += 1
        else:
            if real_backend is None:
                real_backend = TransformersBackend(model_config)
            vector = real_backend(entry, model_config)
            vector = np.asarray(vector, dtype=np.float32)
            validate_acoustic({entry["label"]: vector})
            if use_cache:
                temporary = cache_path.with_name(cache_path.name + f".{uuid.uuid4().hex}.tmp")
                with temporary.open("wb") as handle:
                    pickle.dump(vector, handle, protocol=pickle.HIGHEST_PROTOCOL)
                if cache_path.exists():
                    temporary.unlink()
                else:
                    os.rename(temporary, cache_path)
            computed += 1
        result[entry["label"]] = np.asarray(vector, dtype=np.float32)
    validate_acoustic(result, expected_keys=[row["label"] for row in inventory])
    return result, {"computed": computed, "cached": cached, "errors": 0,
                    "backend": "stub" if backend is not None else "transformers"}


## Bundle validation

Verify the five portable outputs, canonical inventory, vector dimensions, model metadata, and inference counts without source audio.


In [ ]:
REQUIRED={"pinyin_embeddings.pkl","sound_inventory.jsonl","config.json","reports/inference.json"}

def validate_acoustic_bundle(path,*,expected_manifest_sha256=None):
    root=Path(path).resolve();mp=root/"manifest.json"
    if expected_manifest_sha256 and sha256_file(mp)!=expected_manifest_sha256: raise ValueError("acoustic bundle manifest checksum mismatch")
    manifest=json.loads(mp.read_text(encoding="utf-8"))
    if manifest.get("bundle_type")!="acoustic_embeddings" or manifest.get("schema_version")!=1 or manifest.get("data_contract")!=DATA_CONTRACT_VERSION or manifest.get("pinyin_contract") != PINYIN_CONTRACT_VERSION or manifest.get("parser_provenance") != PARSER_PROVENANCE: raise ValueError("unsupported acoustic bundle")
    arts=manifest.get("artifacts",{});
    if not REQUIRED.issubset(arts): raise ValueError("missing acoustic bundle artifacts")
    for rel,digest in arts.items():
        p=safe_artifact(root,rel)
        if not p.is_file() or sha256_file(p)!=digest: raise ValueError(f"artifact checksum mismatch: {rel}")
    inv=read_jsonl(root/"sound_inventory.jsonl")
    if not inv or len({r.get("label") for r in inv})!=len(inv): raise ValueError("invalid acoustic inventory")
    for row in inv:
        label=row.get("label");relative=row.get("relative_path");digest=row.get("sha256")
        if ("path" in row or not isinstance(relative,str) or not relative or Path(relative).is_absolute() or ".." in Path(relative).parts
                or not isinstance(label,str) or canonical_labels([label],tone_convention="explicit")[0]!=label
                or not isinstance(digest,str) or re.fullmatch(r"[0-9a-f]{64}",digest) is None): raise ValueError("inventory is not portable or canonical")
        decompose(label)
    with (root/"pinyin_embeddings.pkl").open("rb") as h:vectors=pickle.load(h)
    dim=validate_acoustic(vectors,expected_keys=[r["label"] for r in inv])
    cfg=json.loads((root/"config.json").read_text(encoding="utf-8"));model=validate_model_config(cfg["model"])
    if cfg.get("bundle_id")!=manifest.get("bundle_id"): raise ValueError("bundle identity mismatch")
    if {**model,"backend":manifest["model"].get("backend")}!=manifest.get("model"): raise ValueError("bundle model metadata mismatch")
    inf=json.loads((root/"reports/inference.json").read_text(encoding="utf-8"))
    for field in ("computed","cached","errors"):
        if not isinstance(inf.get(field),int) or isinstance(inf.get(field),bool) or inf[field]<0: raise ValueError("invalid inference counts")
    if inf.get("backend") not in {"stub","transformers"} or manifest.get("model",{}).get("backend")!=inf.get("backend"): raise ValueError("invalid inference backend")
    if inf!=manifest.get("inference") or inf["computed"]+inf["cached"]!=len(inv) or inf["errors"]!=0: raise ValueError("invalid inference counts")
    for field in ("count","dimension"):
        if not isinstance(manifest.get(field),int) or isinstance(manifest.get(field),bool) or manifest[field]<=0: raise ValueError("bundle counts mismatch")
    if manifest["count"]!=len(inv) or manifest["dimension"]!=dim or manifest.get("inventory_sha256")!=sha256_file(root/"sound_inventory.jsonl"): raise ValueError("bundle counts mismatch")
    return manifest


## Atomic bundle creation

Build the complete inventory, embed it, validate a temporary bundle, and publish without replacing an existing target.


In [ ]:
def prepare_acoustic_bundle(config,*,embedding_backend=None):
    if not isinstance(config,dict) or set(config)!={"bundle_id","output_root","cache_root","sounds","model"}: raise ValueError("invalid acoustic bundle config keys")
    cfg=dict(config); _keys(cfg["sounds"],{"path","tone_convention"},{"sha256","manifest"},"sounds");validate_model_config(cfg["model"])
    if cfg["sounds"]["tone_convention"] not in {"explicit","neutral_unmarked"}: raise ValueError("invalid tone convention")
    bid=cfg["bundle_id"]
    if not isinstance(bid,str) or not bid or Path(bid).name!=bid: raise ValueError("invalid bundle_id")
    out=Path(cfg["output_root"]).resolve();out.mkdir(parents=True,exist_ok=True);target=out/bid
    if target.exists(): raise FileExistsError(target)
    inventory,source=read_sounds(cfg["sounds"])
    for row in inventory: decompose(row["label"])
    vectors,inference=embed_inventory(inventory,cfg["model"],cfg["cache_root"],embedding_backend)
    temp=Path(tempfile.mkdtemp(prefix=f".{bid}.",dir=out))
    try:
        (temp/"reports").mkdir();
        with (temp/"pinyin_embeddings.pkl").open("wb") as h: pickle.dump(vectors,h,pickle.HIGHEST_PROTOCOL)
        portable=[{k:v for k,v in row.items() if k!="path"} for row in inventory]
        write_jsonl(temp/"sound_inventory.jsonl",portable);write_json(temp/"config.json",{"schema_version":1,**cfg})
        write_json(temp/"reports/inference.json",inference)
        artifacts={str(p.relative_to(temp)).replace("\\","/"):sha256_file(p) for p in temp.rglob("*") if p.is_file()}
        env={"python":platform.python_version()}
        for package in ("numpy","torch","transformers"):
            try: env[package]=importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError: pass
        code = dict(BUILD_METADATA["inline_source_sha256"])
        model={**cfg["model"],"backend":inference["backend"]}
        write_json(temp/"manifest.json",{"bundle_type":"acoustic_embeddings","schema_version":SCHEMA_VERSION,"data_contract":DATA_CONTRACT_VERSION,
          "pinyin_contract":PINYIN_CONTRACT_VERSION,"parser_provenance":PARSER_PROVENANCE,"bundle_id":bid,"manifest_self_hash":"excluded","artifacts":artifacts,"source":source,
          "inventory_sha256":sha256_file(temp/"sound_inventory.jsonl"),"count":len(inventory),"dimension":len(next(iter(vectors.values()))),
          "model":model,"inference":inference,"environment":env,"code":code})
        validate_acoustic_bundle(temp);os.rename(temp,target);return target
    except Exception: shutil.rmtree(temp,ignore_errors=True);raise


## Inputs, outputs, and run guard

Set mounted audio/manifest paths, exact hashes and model revision/device.
Upload the contents of local `datasets/source-data` as the Kaggle input dataset: `sounds/`, `sounds-manifest/`, and optionally `duyu-hanzi-pinyin/`. Wav2Vec inference reads only `sounds/` and its manifest.


In [ ]:
DATA_ROOT = Path("/kaggle/input/chinese-sounds-v1")

SOUNDS_DIR = DATA_ROOT / "sounds"
SOUND_MANIFEST_PATH = (
    DATA_ROOT / "sounds-manifest/chinese-sounds-v1/sounds_manifest.csv"
)
SOUND_MANIFEST_SHA256 = (
    "a5c4b7ac719804cc3cfdb853ff9365d673d1a2ca98c8e37d0ea275020533acaa"
)

OUTPUT_ROOT = Path("/kaggle/working/pinyin-embeddings")
CACHE_ROOT = Path("/kaggle/working/acoustic-cache")

MODEL_ID = "jonatasgrosman/wav2vec2-large-xlsr-53-chinese-zh-cn"
MODEL_REVISION = "99ccb2737be22b8bb50dcfcc39ad4d567fb90cfd"
PROCESSOR_ID = MODEL_ID

DEVICE = "cuda"

CONFIG = {
    "bundle_id":"chinese-sounds-v1",
    "output_root":str(OUTPUT_ROOT),
    "cache_root":str(CACHE_ROOT),
    "sounds":{
        "path":str(SOUNDS_DIR),
        "manifest":str(SOUND_MANIFEST_PATH),
        "sha256":SOUND_MANIFEST_SHA256,
        "tone_convention":"explicit"},
        "model":{
            "id":MODEL_ID,
            "revision":MODEL_REVISION,
            "processor":PROCESSOR_ID,
            "layer":-1,
            "pooling":"mean_all",
            "sample_rate":16000,
            "mono":True,
            "padding":"legacy_one_second",
            "dtype":"float32",
            "device":DEVICE
        }
}

In [ ]:
assert "REPLACE_WITH" not in repr(CONFIG)
bundle_path = prepare_acoustic_bundle(CONFIG)
validated = validate_acoustic_bundle(bundle_path)
print(bundle_path)
print("files:", sorted(validated["artifacts"]), "manifest.json")
